In [1]:
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality, get_column_plot
from sdv.metadata import Metadata
import pandas as pd
import os

/tmp/ipykernel_2096590/1097962014.py:1: FutureWarning: The evaluation functions are now accessible via the 'sdv.evaluation' module.
  from sdv.evaluation.single_table import run_diagnostic, evaluate_quality, get_column_plot


In [2]:
# Constants
REAL_DATA_PATH = "../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
METADATA_FILE = "../../DataGeneration/SyntheticData/UAVATTACK/SDV/spoofing_metadata.json"
COLUMN_TO_PLOT = "lat_y"  # Change this to the column you want to plot

In [3]:
# Load real data
real_data = pd.read_csv(REAL_DATA_PATH)

In [4]:
# Data Preprocessing
if 'timestamp' in real_data.columns:
    real_data = real_data.drop(columns=['timestamp'])

# Clean up column names
real_data.columns = real_data.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Encode 'label'
le = None
if 'label' in real_data.columns and (real_data['label'].dtype == object or real_data['label'].dtype.name == 'category'):
    from sklearn.preprocessing import LabelEncoder

    le = LabelEncoder()
    real_data['label'] = le.fit_transform(real_data['label'])


def match_real_schema(synthetic_data):
    synthetic_data = synthetic_data.copy()
    if 'timestamp' in synthetic_data.columns:
        synthetic_data = synthetic_data.drop(columns=['timestamp'])
    synthetic_data.columns = synthetic_data.columns.str.replace('[\\[\\]<>]', '_', regex=True)
    if le is not None and 'label' in synthetic_data.columns and synthetic_data['label'].dtype == object:
        synthetic_data['label'] = le.transform(synthetic_data['label'])
    return synthetic_data

In [5]:
# Load or create metadata
if os.path.exists(METADATA_FILE):
    metadata = Metadata.load_from_json(METADATA_FILE)
    print(f"Loaded metadata from {METADATA_FILE}")
else:
    metadata = Metadata()
    metadata.detect_table_from_dataframe(
        data=real_data,
        table_name='spoofing_data'
    )
    metadata.save_to_json(METADATA_FILE)
    print(f"Created and saved new metadata to {METADATA_FILE}")

Loaded metadata from ../../DataGeneration/SyntheticData/UAVATTACK/SDV/spoofing_metadata.json


In [6]:
# Synthesizer
synthesizers = [
    {
        'name': 'CTGAN',
        'output': "../../Datasets/csv/UAVATTACK/spoofing_ctgan_10000.csv"
    },
    {
        'name': 'GaussianCopula',
        'output': "../../Datasets/csv/UAVATTACK/spoofing_gc_10000.csv"
    },
    {
        'name': 'TVAE',
        'output': "../../Datasets/csv/UAVATTACK/spoofing_tvae_10000.csv"
    },
    {
        'name': 'CGAN',
        'output': "../../Datasets/csv/UAVATTACK/syntheticSpoofing_cgan_data.csv"
    },
    {
        'name': 'GAN',
        'output': "../../Datasets/csv/UAVATTACK/syntheticSpoofing_gan_data.csv"
    }

]

In [7]:
for synth in synthesizers:
    print(f"--- {synth['name']} ---")
    synthetic_data = match_real_schema(pd.read_csv(synth['output']))

    diagnostic = run_diagnostic(real_data, synthetic_data, metadata)

    quality_report = evaluate_quality(real_data, synthetic_data, metadata)

--- CTGAN ---
Generating report ...



|          | 0/83 [00:00<?, ?it/s]|

(1/4) Evaluating Data Validity: |          | 0/83 [00:00<?, ?it/s]|

(1/4) Evaluating Data Validity: |██████████| 83/83 [00:00<00:00, 2250.19it/s]|


Data Validity Score: 100.0%



|          | 0/1 [00:00<?, ?it/s]|

... (588 outputs left out)


Column Pair Trends Score: 78.15%



(3/4) Evaluating Cardinality: N/A
This property does not apply to single-table data.



(4/4) Evaluating Intertable Trends: N/A
This property does not apply to single-table data.



Overall Score (Average): 59.67%



In [8]:
for synth in synthesizers:
    print(f"--- {synth['name']} ---")
    synthetic_data = match_real_schema(pd.read_csv(synth['output']))
    fig = get_column_plot(
        real_data=real_data,
        synthetic_data=synthetic_data,
        metadata=metadata,
        column_name=COLUMN_TO_PLOT
    )
    fig.show()

--- CTGAN ---


--- GaussianCopula ---


--- TVAE ---


--- CGAN ---


--- GAN ---
